# Metadata and Data Dictionary — Freedom in the World

**Notebook 02 of 08**

### Purpose

The raw CSV identifies economies by `REF_AREA` codes and indicators by `FH_FIW_*` codes. This notebook builds the **data dictionary** — the readable lookup tables that turn those codes into names, question texts and scales — and explains where each piece of the dictionary comes from.

### Scope

This notebook only **reads** the raw files; it writes nothing to `data/raw/`. Its outputs — the two lookup tables — are saved to `data/processed/lookup_tables/`.

## Data and Inputs

| Item | Location | Role |
|---|---|---|
| Metadata JSON | `data/raw/FH_FIW.json` | Economy names, citation, license, coverage |
| Raw wide CSV | `data/raw/FH_FIW_WIDEF.csv` | Indicator names and scales (via label columns) |

**Question this notebook answers:** *What does the metadata tell us about the dataset, and how does it map onto the raw CSV?*

**Method:** inspect the metadata JSON section by section, extract the economy lookup from it, build the indicator lookup from the CSV labels, cross-check the two sources, and save the lookup tables.

## Setup: imports and the project root

Same bootstrap as notebook 01: walk up to the project root, then import the reusable helpers. This notebook uses `load_metadata()` and the two dictionary builders from `src/`.

In [ ]:
import sys
from pathlib import Path

current = Path.cwd()
while not (current / 'data' / 'raw' / 'FH_FIW_WIDEF.csv').exists():
    current = current.parent
    if current == current.parent:
        raise RuntimeError('Could not find the project root.')

if str(current) not in sys.path:
    sys.path.insert(0, str(current))

import pandas as pd

from src.data_loader import load_metadata, load_raw_data
from src.countries import get_country_dictionary
from src.indicators import get_indicator_dictionary

metadata = load_metadata()
df = load_raw_data()
print('metadata JSON loaded:', type(metadata).__name__)
print('raw CSV loaded:', df.shape)

metadata JSON loaded: dict
raw CSV loaded: (7880, 53)


### Interpretation

Both raw inputs loaded. `metadata` is a Python dictionary with the JSON's contents; `df` is the same 7880-row wide frame from notebook 01. Now we open the metadata and look at its sections.

## 1. The metadata JSON at a glance

**Question:** what sections does the metadata file contain?

**Method:** list the top-level keys of the JSON.

In [ ]:
print('Top-level sections of the metadata JSON:')
for key in metadata:
    print(' -', key)

Top-level sections of the metadata JSON:
 - schema
 - schema_version
 - type
 - idno
 - changed
 - changed_utc
 - created
 - created_utc
 - created_by
 - changed_by
 - database_description
 - metadata_information


### Interpretation

The file follows a `timeseries-db` schema: a few bookkeeping fields (`schema`, `idno`, `created`, …), then two content sections:

- **`database_description`** — the actual data dictionary: title, abstract, citation, license, time coverage, topics, and the economy list (`ref_country`).
- **`metadata_information`** — provenance of the metadata itself (who produced it, version).

We focus on `database_description`.

## 2. The database description

**Question:** what does `database_description` tell us about the dataset as a whole?

**Method:** print the key fields: title, abstract, frequency, coverage, license and citation.

In [ ]:
desc = metadata['database_description']

print('Title:', desc['title_statement']['title'])
print('Abstract:', desc['abstract'][:180], '...')
print('Update frequency:', desc['update_frequency'])
coverage = desc['time_coverage'][0]
print('Time coverage:', coverage['start'], '-', coverage['end'])
print('License:', desc['license'][0]['name'])
print('Citation:', desc['citation'])

Title: Freedom in the World
Abstract: Freedom in the World, produced by Freedom House, is an annual report assessing political rights and civil liberties in 195 countries and 13 territories. It uses numerical ratings a ...
Update frequency: Annual
Time coverage: 2013 - 2026
License: License Specified Externally
Citation: Freedom House. (Year). Freedom in the World Year. Retrieved from https://freedomhouse.org/report/freedom-world#Data


### Interpretation

The description confirms what notebook 01 found: the dataset is Freedom House's **Freedom in the World** assessment, updated **annually**, covering **2013–2026**. It also gives the citation template and the license (non-commercial use with attribution) — both belong in the README and the dashboard's methodology page.

## 3. What the JSON does NOT contain

**Question:** where do the indicator definitions live?

**Method:** look for lookup-like sections in the JSON, then contrast with the CSV.

In [ ]:
lookup_sections = [k for k in desc.keys() if 'country' in k or 'indicator' in k or 'series' in k]
print('Lookup-like sections inside database_description:', lookup_sections)
print('Indicator codes defined in the CSV:', df['INDICATOR'].nunique())
print()
print('Indicator info available in the CSV columns:')
print([c for c in df.columns if 'INDICATOR' in c or 'UNIT_MEASURE' in c])

Lookup-like sections inside database_description: ['ref_country']
Indicator codes defined in the CSV: 40

Indicator info available in the CSV columns:
['INDICATOR', 'UNIT_MEASURE', 'INDICATOR_LABEL', 'UNIT_MEASURE_LABEL']


### Interpretation

A key finding: **the metadata JSON contains no indicator definitions.** Its only lookup section is `ref_country` (economies). The indicator names, question texts and scales live in the CSV itself — in the `INDICATOR_LABEL`, `UNIT_MEASURE` and `UNIT_MEASURE_LABEL` columns. The data dictionary therefore has **two sources**: the JSON for economies, the CSV for indicators.

## 4. The economy lookup table

**Question:** what does `ref_country` give us?

**Method:** build the lookup with `get_country_dictionary()` (reads `ref_country`) and preview it.

In [ ]:
countries = get_country_dictionary()
print('Economy lookup shape:', countries.shape)
countries.head(10)

Economy lookup shape: (197, 2)


,REF_AREA,Economy
0,SLB,Solomon Islands
1,SLV,El Salvador
2,MOZ,Mozambique
3,KNA,St. Kitts and Nevis
4,LSO,Lesotho
5,MLT,Malta
6,PRT,Portugal
7,MNG,Mongolia
8,PER,Peru
9,GMB,"Gambia, The"


### Interpretation

The JSON provides a clean **code → name** lookup: **197 economies**, each with a `REF_AREA` code and its official Data360 name (e.g. `XKX` → Kosovo, `TWN` → Taiwan, China). This is the authoritative source for economy names — the notebook never invents a name.

## 5. Cross-check: do the CSV and the JSON agree?

**Question:** is the economy lookup complete and consistent with the raw CSV?

**Method:** compare the sets of codes in both files, in both directions.

In [6]:
csv_codes = set(df['REF_AREA'].unique())
json_codes = set(countries['REF_AREA'])

print('Codes in the CSV:', len(csv_codes))
print('Codes in the JSON lookup:', len(json_codes))
print('In the CSV but missing from the JSON:', sorted(csv_codes - json_codes))
print('In the JSON but absent from the CSV:', sorted(json_codes - csv_codes))

Codes in the CSV: 197
Codes in the JSON lookup: 197
In the CSV but missing from the JSON: []
In the JSON but absent from the CSV: []


### Interpretation

The two sources agree **exactly**: every one of the 197 codes in the CSV resolves through the JSON lookup, and the lookup contains nothing that is not in the CSV. No codes need fixing or guessing — the economy lookup is complete as-is.

## 6. The indicator lookup table

**Question:** what does the indicator dictionary look like?

**Method:** build it with `get_indicator_dictionary()` (from the CSV label columns) and preview it.

In [ ]:
indicators = get_indicator_dictionary()
print('Indicator lookup shape:', indicators.shape)
indicators.head(8)

Indicator lookup shape: (40, 5)


,INDICATOR,INDICATOR_LABEL,Category,UNIT_MEASURE,UNIT_MEASURE_LABEL
0,FH_FIW_F3,Rule of Law: Is there protection from the ille...,Rule of Law,0_TO_4,0-4 scale
1,FH_FIW_F4,"Rule of Law: Do laws, policies, and practices ...",Rule of Law,0_TO_4,0-4 scale
2,FH_FIW_G2,Personal Autonomy And Individual Rights: Are i...,Personal Autonomy And Individual Rights,0_TO_4,0-4 scale
3,FH_FIW_G3,Personal Autonomy And Individual Rights: Do in...,Personal Autonomy And Individual Rights,0_TO_4,0-4 scale
4,FH_FIW_G4,Personal Autonomy And Individual Rights: Do in...,Personal Autonomy And Individual Rights,0_TO_4,0-4 scale
5,FH_FIW_CL,Freedom in the World: Civil liberties scores,Freedom in the World,0_TO_60,0-60 scale
6,FH_FIW_D2,Freedom of Expression and Belief: Are individu...,Freedom of Expression and Belief,0_TO_4,0-4 scale
7,FH_FIW_D3,Freedom of Expression and Belief: Is there aca...,Freedom of Expression and Belief,0_TO_4,0-4 scale


### Interpretation

The indicator lookup holds **40 rows** — one per indicator code — with the question text (`INDICATOR_LABEL`), a `Category`, and the scale (`UNIT_MEASURE` / `UNIT_MEASURE_LABEL`). The question text is the survey question verbatim; the scale is the same `UNIT_MEASURE` column we saw in notebook 01 (0–4 up to 0–100, the 1–7 ratings, and the categorical status).

## 7. Where the Category comes from

**Question:** how is the `Category` column derived?

**Method:** the label text itself carries the category as its prefix — everything before the first `": "`. We demonstrate the rule on a sample.

In [8]:
samples = indicators[['INDICATOR', 'INDICATOR_LABEL', 'Category']].head(5).copy()
samples

,INDICATOR,INDICATOR_LABEL,Category
0,FH_FIW_F3,Rule of Law: Is there protection from the ille...,Rule of Law
1,FH_FIW_F4,"Rule of Law: Do laws, policies, and practices ...",Rule of Law
2,FH_FIW_G2,Personal Autonomy And Individual Rights: Are i...,Personal Autonomy And Individual Rights
3,FH_FIW_G3,Personal Autonomy And Individual Rights: Do in...,Personal Autonomy And Individual Rights
4,FH_FIW_G4,Personal Autonomy And Individual Rights: Do in...,Personal Autonomy And Individual Rights


In [ ]:
print('Category distribution across the 40 indicators:')
print(indicators['Category'].value_counts().to_string())

Category distribution across the 40 indicators:
Category
Freedom in the World                       6
Rule of Law                                4
Personal Autonomy And Individual Rights    4
Freedom of Expression and Belief           4
Civil liberties                            4
Political Pluralism and Participation      4
Associational and Organizational Rights    3
Political rights                           3
Functioning of Government                  3
Electoral Process                          3
Political rights, additional Question Q    1
Political rights, additional Question A    1


### Interpretation

The category is the label segment before the first `": "` — e.g. `Rule of Law: Is there protection ...` becomes `Rule of Law`. The 40 indicators fall into **12 distinct prefixes**: the 7 question categories (Electoral Process, Political Pluralism and Participation, Functioning of Government, Freedom of Expression and Belief, Associational and Organizational Rights, Rule of Law, Personal Autonomy and Individual Rights), the 2 subtotal prefixes (Political rights, Civil liberties), 2 additional-question prefixes, and the summary prefix `Freedom in the World` (TOTAL, PR, CL, the two ratings and STATUS — 6 indicators). Every category string is written in the CSV itself — nothing is invented.

## 8. Mapping the lookups onto the raw CSV

**Question:** how do the two lookups resolve the raw codes?

**Method:** join a few raw rows with the economy and indicator lookups.

In [ ]:
mapped = df[['REF_AREA', 'INDICATOR']].head(5).merge(
    countries, on='REF_AREA', how='left'
).merge(
    indicators[['INDICATOR', 'INDICATOR_LABEL']], on='INDICATOR', how='left'
)
mapped

,REF_AREA,INDICATOR,Economy,INDICATOR_LABEL
0,COD,FH_FIW_F3,"Congo, Dem. Rep.",Rule of Law: Is there protection from the ille...
1,MYS,FH_FIW_F3,Malaysia,Rule of Law: Is there protection from the ille...
2,TZA,FH_FIW_F4,Tanzania,"Rule of Law: Do laws, policies, and practices ..."
3,TZA,FH_FIW_G2,Tanzania,Personal Autonomy And Individual Rights: Are i...
4,BEL,FH_FIW_G3,Belgium,Personal Autonomy And Individual Rights: Do in...


### Interpretation

The raw codes now read as plain language: `TZA` + `FH_FIW_F4` becomes *Tanzania — Do laws, policies, and practices guarantee equal treatment ...?*. This is exactly the mapping notebook 03 will use when it builds the cleaned long-format dataset: the raw CSV stays untouched, and the lookups add the readable names.

## 9. Save the lookup tables

**Question:** where do the dictionaries live for the rest of the pipeline?

**Method:** write both lookups as CSV files under `data/processed/lookup_tables/`.

In [ ]:
out_dir = current / 'data' / 'processed' / 'lookup_tables'
out_dir.mkdir(parents=True, exist_ok=True)

countries.to_csv(out_dir / 'economy_lookup.csv', index=False)
indicators.to_csv(out_dir / 'indicator_lookup.csv', index=False)

print('Saved', out_dir / 'economy_lookup.csv', '| rows:', len(countries))
print('Saved', out_dir / 'indicator_lookup.csv', '| rows:', len(indicators))

Saved C:\Users\OgwalJoshuaRobin\OneDrive - War Child\Desktop\Freedom\data\processed\lookup_tables\economy_lookup.csv | rows: 197
Saved C:\Users\OgwalJoshuaRobin\OneDrive - War Child\Desktop\Freedom\data\processed\lookup_tables\indicator_lookup.csv | rows: 40


## Summary and next question

### What we learned

- The metadata JSON's `database_description` carries the dataset description, citation, license and coverage — and its only lookup is **`ref_country`**: 197 codes that match the CSV **exactly**.
- **Indicator definitions are not in the JSON** — they come from the CSV's own label columns. The indicator lookup has 40 rows: code, question text, category (derived from the label prefix) and scale.
- Both lookup tables are saved to `data/processed/lookup_tables/` for the rest of the pipeline.

### Next question

*How do we turn the wide raw CSV into the clean long-format dataset?* — that is notebook 03, data cleaning and preparation, which joins these lookups in as it reshapes the data.